# 🚀 Video2Video – Lanzador Optimizado para Kaggle (GPU Dual T4 ×2)

### ⚙️ Configuración previa en la barra lateral derecha de Kaggle:
1. **Accelerator**: Selecciona **GPU T4 x2** (te otorga 32 GB VRAM combinada).
2. **Internet**: Asegúrate de que esté en **Internet on** (requiere verificación telefónica gratuita de Kaggle).
3. **Secrets**: En `Add-ons` -> `Secrets`, agrega tus claves:
   - `GHP_GITHUB_TOKEN` *(opcional si el repo es público, obligatorio si es privado)*
   - `GOOGLE_API_KEY` *(para Gemini Flash / traducción y títulos virales)*
   - `FISH_API_KEY` *(para síntesis de voz Fish Audio TTS)*

   - `HF_TOKEN` *(obligatorio para Pyannote Diarization 3.1)*

In [ ]:
# @title ⚡ Arrancar V2V + OpenReel (2x T4 + Cloudflare Tunnel)
import os, time, threading, sys, warnings, subprocess, re, urllib.request
from pathlib import Path
warnings.filterwarnings("ignore")

print("=" * 60)
print("🎬 INICIANDO VIDEO2VIDEO EN KAGGLE (DUAL T4 ACCELERATOR)")
print("=" * 60)

# --- 1. HARDWARE & DUAL-GPU ---
try:
    import torch
    gpu_count = torch.cuda.device_count()
    print(f"🔥 GPUs NVIDIA detectadas: {gpu_count}")
    for i in range(gpu_count):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")
    os.environ["V2V_DEVICE"] = "cuda:0"
    os.environ["V2V_DEVICE_SECONDARY"] = "cuda:1" if gpu_count >= 2 else "cuda:0"
    print("⚡ Modo Dual-GPU activado" if gpu_count >= 2 else "⚡ Modo Single-GPU activado")
except Exception as e:
    print("⚠️ Error comprobando GPUs:", e)

# --- 2. SECRETOS DE KAGGLE ---
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for name in ["GHP_GITHUB_TOKEN", "FISH_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"]:
        val = user_secrets.get_secret(name)
        if val:
            val_str = str(val).strip()
            os.environ[name] = val_str
            masked = f"...{val_str[-4:]}" if len(val_str) >= 4 else "***"
            print(f"🔑 Secreto cargado: {name} ({masked})")
    if os.getenv("HF_TOKEN"):
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"], write_permission=False)
        print("🤗 Hugging Face autenticado con éxito.")
except Exception:
    pass

# --- 3. REPOSITORIO & VARIABLES ---
REPO_PATH = Path("/kaggle/working/v2v")
GHP_TOKEN = os.getenv("GHP_GITHUB_TOKEN", "")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git" if GHP_TOKEN else "https://github.com/almagomen/v2v.git"
BRANCH_NAME = "monorepo-subtree"
BACKEND_PORT, FRONTEND_PORT = 8000, 5173

if not REPO_PATH.exists():
    print("📥 Clonando repositorio en /kaggle/working/v2v...")
    subprocess.run(["git", "clone", "-q", "-b", BRANCH_NAME, REPO_URL, str(REPO_PATH)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", "HEAD", "-q"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "pull", "origin", BRANCH_NAME, "-q"], check=True)
    print("✔️ Repositorio actualizado.")

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))


# Sincronizar secretos en .env local para Vite
env_lines = [f"{k}={os.getenv(k)}\n" for k in ["GOOGLE_API_KEY", "FISH_API_KEY", "HF_TOKEN", "GHP_GITHUB_TOKEN"] if os.getenv(k)]
if env_lines:
    try:
        (REPO_PATH / ".env").write_text("".join(env_lines), encoding="utf-8")
    except Exception:
        pass

# --- 4. DEPENDENCIAS Y MODELOS EN GPU ---
if 'models_loaded' not in globals():
    print("📦 Instalando dependencias con uv...")
    subprocess.run("pip install -q uv && npm install -g pnpm --silent >/dev/null 2>&1 && (which deno >/dev/null || (curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1 && cp -f /root/.deno/bin/deno /usr/local/bin/))", shell=True)

    # 1. Paquetes con dependencias
    subprocess.run(
        "uv pip install 'yt-dlp[default]' bs-roformer-infer tinydb sentencex soundfile "
        "'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' "
        "'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' "
        "'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' "
        "'typing-extensions>=4.12.2' 'google-genai>=2.9.0' -q",
        shell=True,
    )

    # 2. Paquetes con --no-deps (evitan conflictos con PyTorch preinstalado en Kaggle)
    subprocess.run(
        "uv pip install --no-deps git+https://github.com/jianfch/stable-ts.git "
        "git+https://github.com/pyannote/pyannote-audio.git openai-whisper "
        "'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' "
        "'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' -q",
        shell=True,
    )

    # Binario Cloudflare Tunnel
    cf_bin = Path("/tmp/cloudflared")
    if not cf_bin.exists():
        print("🌐 Descargando Cloudflare Tunnel...")
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(cf_bin)], check=True)
        subprocess.run(["chmod", "+x", str(cf_bin)], check=True)

    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model

    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': get_roformer_model(),
        'models_loaded': True
    })
    print("✅ Modelos inicializados en memoria.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 5. SERVIDOR BACKEND (Uvicorn en Background) ---
import uvicorn
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    global server
    for m in [m for m in list(sys.modules.keys()) if m.startswith(('api', 'src')) and 'models' not in m]:
        del sys.modules[m]

    from api.main import app
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_level="warning", loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config)
    server.run()

if 'server' in globals() and server.started:
    print("🛑 Reiniciando servidor backend...")
    server.should_exit = True
    time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# --- 6. FRONTEND (Vite / OpenReel) ---
OPENREEL_PATH = REPO_PATH / "openreel"
WEB_PATH = OPENREEL_PATH / "apps" / "web"

vite_already_running = False
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=1) as resp:
        if resp.status in (200, 304):
            vite_already_running = True
except Exception:
    pass

if vite_already_running and globals().get('front_proc') and globals()['front_proc'].poll() is None:
    print("⚡ OpenReel (Vite) ya está activo. Reutilizando proceso.")
else:
    print("🏗️ Preparando OpenReel...")
    subprocess.run(["pnpm", "install", "--silent"], cwd=str(OPENREEL_PATH), check=True)
    subprocess.run(["fuser", "-k", f"{FRONTEND_PORT}/tcp"], stderr=subprocess.DEVNULL)
    
    globals()['front_proc'] = subprocess.Popen(
        ["pnpm", "exec", "vite", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
        cwd=str(WEB_PATH), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    # Healthcheck Vite
    print("⏳ Esperando que OpenReel responda en el puerto 5173...")
    for _ in range(45):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=1) as resp:
                if resp.status in (200, 304):
                    print("✅ OpenReel Vite activo.")
                    break
        except Exception:
            pass
        time.sleep(1)

# --- 7. CLOUDFLARE TUNNEL (PERSISTENCIA TOTAL A NIVEL DE SO) ---
CF_URL_FILE = Path("/tmp/v2v_tunnel_url.txt")

def check_cf_os_running():
    try:
        r = subprocess.run(["pgrep", "-f", "cloudflared tunnel"], capture_output=True, text=True)
        return r.returncode == 0
    except Exception:
        return False

# Preservar el túnel si cloudflared sigue corriendo en Linux y tenemos su URL guardada
is_running = check_cf_os_running()
has_saved_url = CF_URL_FILE.exists() and bool(CF_URL_FILE.read_text(encoding="utf-8").strip())

if is_running and has_saved_url:
    tunnel_url = CF_URL_FILE.read_text(encoding="utf-8").strip()
    globals()['tunnel_url'] = tunnel_url
    print(f"♻️ Preservando túnel Cloudflare activo en el sistema: {tunnel_url}")
    print("   (La URL no cambia entre re-ejecuciones para no perder tu timeline ni datos locales)")
else:
    print("🌐 Levantando túnel HTTPS público para OpenReel...")
    subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
    time.sleep(0.5)
    cf_proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{FRONTEND_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    tunnel_url = None
    for _ in range(40):
        line = cf_proc.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(0.5)

    # Drenar Cloudflare stdout en background sin bloquear
    threading.Thread(target=lambda: [None for _ in iter(cf_proc.stdout.readline, '')], daemon=True).start()
    globals()['cf_proc'] = cf_proc
    globals()['tunnel_url'] = tunnel_url
    if tunnel_url:
        CF_URL_FILE.write_text(tunnel_url, encoding="utf-8")

print("\n" + "=" * 60)
if tunnel_url:
    print("🎉 ¡ENTORNO LISTO Y FUNCIONANDO EN SEGUNDO PLANO!")
    print("🔗 Abre OpenReel en tu navegador:")
    print(f"👉 {tunnel_url}")
else:
    print("⚠️ No se pudo obtener la URL de Cloudflare automáticamente.")
print("=" * 60)


---
## 🔬 Prueba Paso a Paso del Pipeline (Diagnóstico Función por Función)
Ejecuta las siguientes celdas de forma individual para ver los logs que genera cada función exactamente igual que en el panel de V2V, midiendo los tiempos de cada etapa.

In [ ]:
# @title 🔗 0. Configurar URL a Probar
url = "https://www.youtube.com/watch?v=C0XGm2m5Yf4"  # @param {type:"string"}
print(f"🎯 URL seleccionada: {url}")


In [ ]:
# @title 📥 1. Descarga y Separación de Stems (Control Interactivo & Benchmark)
# @markdown El solapamiento está calibrado internamente en 2.0s con crossfade para garantizar una unión perfecta sin 'tics'.
modo_gpu = "Dual-GPU (GPU 0 + GPU 1)" # @param ["Dual-GPU (GPU 0 + GPU 1)", "Single-GPU (1 sola GPU)"]
forzar_reproceso = True # @param {type:"boolean"}

import time, os
from pathlib import Path
import soundfile as sf
import IPython.display as ipd
from src.download import download_video, separate_vocals

force_dual = (modo_gpu == "Dual-GPU (GPU 0 + GPU 1)")

print(f"⚙️ Analizando video: {url}")
video_path, info = download_video(url, force_new=False)

vocals_path = video_path.with_name("vocals.wav")
no_vocals_path = video_path.with_name("no_vocals.wav")

if forzar_reproceso:
    if vocals_path.exists(): vocals_path.unlink()
    if no_vocals_path.exists(): no_vocals_path.unlink()

print("=" * 60)
print("🎬 Iniciando separación con:")
print(f"   • Modo GPU: {'⚡ Dual-GPU (GPU 0 + GPU 1)' if force_dual else '🛡️ Single-GPU'}")
print("   • Solapamiento: 2.0s (fijo con crossfade anti-tic)")
print("=" * 60)

t0 = time.perf_counter()
voc_path, inst_path = separate_vocals(
    video_path,
    force_dual_gpu=force_dual,
)
t_elapsed = time.perf_counter() - t0

# Métricas usando soundfile sobre el archivo WAV extraído
audio_info = sf.info(str(voc_path))
dur_sec = audio_info.duration
sr = audio_info.samplerate
total_frames = audio_info.frames
speedup = dur_sec / t_elapsed if t_elapsed > 0 else 0

print("\n" + "=" * 60)
print("🏁 RESULTADOS DEL BENCHMARK:")
print(f"• Duración del audio: {dur_sec:.1f}s ({dur_sec/60:.2f} min)")
print(f"• Tiempo de separación: {t_elapsed:.1f} segundos")
print(f"• Velocidad relativa: {speedup:.1f}x tiempo real")
print("=" * 60)

# Reproductores interactivos de audio directamente desde disco
print("\n🎵 1. Pista de Voz Extraída (vocals.wav):")
ipd.display(ipd.Audio(str(voc_path)))

# Zoom de 10s en el punto de unión (midpoint) para verificar la transición limpia
if force_dual and dur_sec > 20:
    mid_sample = total_frames // 2
    win_samples = int(5.0 * sr)
    start_s = max(0, mid_sample - win_samples)
    end_s = min(total_frames, mid_sample + win_samples)
    zoom_data, _ = sf.read(str(voc_path), start=start_s, stop=end_s)
    if zoom_data.ndim == 2:
        zoom_data = zoom_data.T
    mid_sec = dur_sec / 2.0
    print(f"\n🔍 2. Zoom en el punto de unión (segundo {mid_sec-5:.1f} a {mid_sec+5:.1f}):")
    print("   (Verificación auditiva del empalme suave sin cortes ni tics):")
    ipd.display(ipd.Audio(zoom_data, rate=sr))

print("\n🎸 3. Pista Instrumental sin Voz (no_vocals.wav):")
ipd.display(ipd.Audio(str(inst_path)))


In [ ]:
# @title 📝 2. Transcripción (Whisper) y Diarización (Pyannote)
import time
from src.stt import transcribe_and_process

vocals_path = video_path.with_name("vocals.wav")
print(f"📝 Procesando STT + Diarización sobre: {vocals_path.name}\n")
t0 = time.perf_counter()
res_tuple = transcribe_and_process(vocals_path)
result, transcription, diarization = res_tuple[0], res_tuple[1], res_tuple[2]
t_step2 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 2 completado en {t_step2:.1f}s")
print(f"📊 Total segmentos detectados: {len(result)}")
print(f"🌐 Idioma original: {getattr(transcription, 'language', 'desconocido')}")
if result:
    print(f"🔹 Primer segmento: [{result[0]['start']:.2f}s - {result[0]['end']:.2f}s] {result[0]['speaker']}: {result[0]['text']}")
print("=" * 60)


In [ ]:
# @title 🌐 3. Traducción y Adaptación de Diálogos (Gemini)
import time
from src.translation import translate_text_list

texts_to_translate = [r["text"] for r in result]
src_lang = getattr(transcription, "language", "en")
print(f"🌐 Enviando {len(texts_to_translate)} frases a traducir ({src_lang} -> es)...\n")

t0 = time.perf_counter()
translated_texts = translate_text_list(
    texts_to_translate,
    target_lang="es",
    source_lang=src_lang,
    print_full_prompt=True
)
t_step3 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 3 completado en {t_step3:.1f}s")
if translated_texts:
    print(f"✨ Frases adaptadas con éxito: {len(translated_texts)}")
    print(f"   [Original 1]: {texts_to_translate[0]}")
    print(f"   [Adaptada 1]: {translated_texts[0]}")
else:
    print("❌ Falló la traducción.")
print("=" * 60)


In [ ]:
# @title ✨ 4. Generación de Título Viral (Gemini)
import time
from src.translation import generate_tiktok_title

# @markdown **Opciones de entrada:**
# @markdown Elige si generar el título basándote en los **diálogos** (transcripción completa) o en la **URL de YouTube**:
input_source = "dialogues"  # @param ["dialogues", "url"]
show_prompt = True  # @param {type:"boolean"}

if input_source == "dialogues":
    fallback_summary = " ".join(translated_texts) if translated_texts else " ".join([r["text"] for r in result])
    selected_url = None
else:
    fallback_summary = None
    selected_url = url

print(f"✨ Generando título viral para TikTok (fuente: {input_source})...\n")

t0 = time.perf_counter()
viral_title = generate_tiktok_title(
    youtube_url=selected_url,
    target_lang="es",
    fallback_text=fallback_summary,
    print_full_prompt=show_prompt
)
t_step4 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 4 completado en {t_step4:.1f}s")
print(f"🔥 Título viral resultante:\n👉 {viral_title}")
print("=" * 60)


In [ ]:
# @title ✂️ 5. Corte de Secuencias de Audio (Vocals EN)
import time
from src.pipeline import split_audio_sequences

# Estructurar segmentos con el texto traducido
combined_segments = []
tr_list = translated_texts if translated_texts else [r["text"] for r in result]
for orig, tr_text in zip(result, tr_list):
    seg = dict(orig)
    seg["text"] = tr_text
    combined_segments.append(seg)

print(f"✂️ Cortando {len(combined_segments)} clips de audio en Seqs_src...\n")
t0 = time.perf_counter()
split_paths = split_audio_sequences(vocals_path, combined_segments)
t_step5 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 5 completado en {t_step5:.1f}s")
print(f"✂️ Archivos generados: {len(split_paths)} en {vocals_path.parent / 'Seqs_src'}")
print("=" * 60)


In [ ]:
# @title 🗣️ 6. Síntesis de Voz TTS y Ajuste de Tempo (Fish Audio + FFmpeg)
import time
from src.audio_utils import clone_voice_segments
from src.tts import synthesize_speech

speaker_audio = video_path.with_name("vocals.wav")
print(f"🗣️ Iniciando síntesis TTS para {len(combined_segments)} segmentos...\n")

t0 = time.perf_counter()
paths_es, times, rate = clone_voice_segments(
    video_path,
    combined_segments,
    "es",
    speaker_audio,
    synthesize_speech,
    slowdown_range=(0.5, 0.8),
    stretch_target_ratio=0.85
)
t_step6 = time.perf_counter() - t0

print("\n" + "=" * 60)
print(f"✅ Paso 6 completado en {t_step6:.1f}s")
print(f"🗣️ Audios generados en Seqs_trg: {len(paths_es)}")
print("=" * 60)
